In [2]:
from pathlib import Path
import json
import os
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# AWS S3 CONFIGURATION
# ============================================================
import boto3
import shutil

BUCKET = "fire2air-darwin-dan5-646468992009-ap-southeast-2-an"

NTEPA_RAW = f"s3://{BUCKET}/raw/ntepa/"
FIRMS_RAW = f"s3://{BUCKET}/raw/firms/"
NTEPA_DAILY = f"s3://{BUCKET}/processed/ntepa_daily/"
FIRMS_DAILY = f"s3://{BUCKET}/processed/firms_daily/"
MODEL_READY = f"s3://{BUCKET}/processed/model_ready/"
MODEL1_DIR = f"s3://{BUCKET}/models/model1/"
MODEL2_DIR = f"s3://{BUCKET}/models/model2/"
MODEL3_DIR = f"s3://{BUCKET}/models/model3/"
MODEL1_OUTPUT = f"s3://{BUCKET}/outputs/model1/"
MODEL2_OUTPUT = f"s3://{BUCKET}/outputs/model2/"
MODEL3_OUTPUT = f"s3://{BUCKET}/outputs/model3/"
VIS_OUTPUT = f"s3://{BUCKET}/outputs/visualisation/"
DASHBOARD_DIR = f"s3://{BUCKET}/dashboard/"
DOCUMENTATION_DIR = f"s3://{BUCKET}/documentation/"
ATHENA_RESULTS = f"s3://{BUCKET}/athena-results/"

s3 = boto3.client("s3")
LOCAL_ROOT = Path("/tmp/fire2air_a3")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def _prefix(uri):
    return uri.replace(f"s3://{BUCKET}/", "", 1)

def download_prefix(uri, local_dir, flatten=False):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    prefix = _prefix(uri)
    paginator = s3.get_paginator("list_objects_v2")
    downloaded = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            rel = Path(key[len(prefix):].lstrip("/"))
            dest = local_dir / (rel.name if flatten else rel)
            dest.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(BUCKET, key, str(dest))
            downloaded.append(dest)
    return downloaded

def upload_file(local_path, uri, object_name=None):
    local_path = Path(local_path)
    prefix = _prefix(uri).rstrip("/")
    key = f"{prefix}/{object_name or local_path.name}"
    s3.upload_file(str(local_path), BUCKET, key)
    print(f"Uploaded: s3://{BUCKET}/{key}")

def upload_tree(local_dir, uri):
    local_dir = Path(local_dir)
    if not local_dir.exists():
        return
    prefix = _prefix(uri).rstrip("/")
    for p in local_dir.rglob("*"):
        if p.is_file():
            rel = p.relative_to(local_dir).as_posix()
            key = f"{prefix}/{rel}"
            s3.upload_file(str(p), BUCKET, key)
    print(f"Uploaded folder: {local_dir} -> {uri}")

RANDOM_STATE = 661
PROJECT_YEARS = list(range(2018, 2025))
PM25_THRESHOLD = 25.0
MIN_VALID_PM_HOURS = 18
MIN_VALID_COUNT_HOURS = 24
MAX_FIRE_DISTANCE_KM = 500.0
PREFILTER_DISTANCE_KM = 520.0
UPWIND_ANGLE_DEG = 45.0

# Transparent FSI research parameters. These are engineering defaults, not physical laws.
FSI_DISTANCE_TAU_KM = 100.0
FSI_AGE_HALF_LIFE_HOURS = 24.0
FSI_CONFIDENCE_WEIGHTS = {"l": 0.50, "n": 0.75, "h": 1.00}

# Optional future API acquisition. Historical Assessment 3 modelling continues to use
# the supplied 2018–2024 archive files so the evaluation period is reproducible.
USE_FIRMS_API = False
FIRMS_API_SOURCE = os.getenv("FIRMS_API_SOURCE", "VIIRS_SNPP_NRT")
FIRMS_API_BBOX = "125.5,-17.5,136.5,-7.5"  # west,south,east,north

# AWS local staging workspace. Raw files are downloaded from S3 for processing,
# while final checkpoints are uploaded back to the shared S3 prefixes.
project_folder = LOCAL_ROOT / "processing_workspace"
if project_folder.exists():
    shutil.rmtree(project_folder)
project_folder.mkdir(parents=True, exist_ok=True)

print("Downloading raw NT EPA data from:", NTEPA_RAW)
download_prefix(NTEPA_RAW, project_folder / "ntepa")
print("Downloading raw FIRMS data from:", FIRMS_RAW)
download_prefix(FIRMS_RAW, project_folder / "firms")

output_root = project_folder / "outputs_prt661_a3"
table_dir = output_root / "tables"
processed_dir = output_root / "processed"
for folder in [output_root, table_dir, processed_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("AWS staging folder:", project_folder)


AWS staging folder: \tmp\fire2air_a3\processing_workspace
